## 1 - Make the main list

In [21]:
#create only the main list
import json
import re
from pathlib import Path
from typing import List
import pandas as pd

# =========================
# CONFIGURE THESE PATHS
# =========================
SNAPSHOT_DIR = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots")
OUTPUT_DIR   = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label\Method_V2.0")
MAIN_COMMIT_LIST = OUTPUT_DIR / "Main_Commit_List.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# NORMALIZATION / HELPERS
# =========================
URL_RE = re.compile(r"http\S+|www\.\S+", re.I)

def normalize_subject(s: str) -> str:
    """Lowercase, strip URLs/emojis/control chars, collapse whitespace."""
    s = (s or "").lower()
    s = URL_RE.sub("", s)
    s = re.sub(r"[\u0000-\u001f\u007f]", " ", s)                 # control chars
    s = re.sub(r"[\U00010000-\U0010FFFF]", "", s)                # emojis / astral symbols
    s = re.sub(r"\s+", " ", s).strip()                           # collapse spaces
    return s

def load_snapshots(folder: Path) -> pd.DataFrame:
    """
    Read all .jsonl files and dedupe to one row per (repo, sha).
    Keep 'subject_raw' and a normalized 'subject_norm'.
    """
    rows = []
    for p in folder.glob("*.jsonl"):
        with p.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    d = json.loads(line)
                except Exception:
                    continue
                repo = d.get("repo") or p.stem
                sha  = d.get("sha")
                if not sha:
                    continue
                subj = d.get("subject") or d.get("commit_raw") or d.get("message") or ""
                rows.append({"repo": repo, "sha": sha, "subject_raw": subj})

    if not rows:
        return pd.DataFrame(columns=["repo", "sha", "subject_raw", "subject_norm"])

    df = pd.DataFrame(rows)
    # Prefer rows with non-empty subject if duplicates
    df["has_subj"] = df["subject_raw"].fillna("").ne("")
    df = (
        df.sort_values(["repo", "sha", "has_subj"], ascending=[True, True, False])
          .drop_duplicates(subset=["repo", "sha"], keep="first")
          .drop(columns=["has_subj"])
    )
    df["subject_norm"] = df["subject_raw"].apply(normalize_subject)
    return df

# =========================
# MAIN
# =========================
def main():
    df = load_snapshots(SNAPSHOT_DIR)
    print(f"[INFO] Commits loaded (deduped): {len(df)}")

    # Minimal commit list only
    cols = ["repo", "sha", "subject_raw", "subject_norm"]
    df[cols].to_csv(MAIN_COMMIT_LIST, index=False, encoding="utf-8")
    print(f"[OK] Wrote main commit list: {MAIN_COMMIT_LIST} (rows={len(df)})")

if __name__ == "__main__":
    main()



[INFO] Commits loaded (deduped): 106597
[OK] Wrote main commit list: C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label\Method_V2.0\Main_Commit_List.csv (rows=106597)


## 2 - Generate the 70/30 sampling from all commits

In [3]:
import pandas as pd
import random
from pathlib import Path

# =========================
# CONFIG
# =========================
BASE_DIR   = Path(r"C:\Thesis_Temp")
INPUT_CSV  = BASE_DIR / "Main_Commit_List.csv"       # attached main list
OUTPUT_DIR = BASE_DIR / "Second_Label_Intent"

SAMPLE_SIZE        = 1000    # total size across DEV+TEST
SEED               = 12345   # fixed seed for reproducibility
STRATIFY_BY_REPO   = False   # True = proportional per repo
DEV_RATIO          = 0.70    # 70% to DEV, 30% to TEST

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# HELPERS
# =========================
def sample_rows(df_in: pd.DataFrame, n: int, seed: int, stratify: bool) -> pd.DataFrame:
    """Sample n rows from df_in (optionally stratified by repo)."""
    if df_in.empty:
        return df_in
    n = min(n, len(df_in))
    if not stratify or "repo" not in df_in.columns:
        return df_in.sample(n=n, random_state=seed)

    # Proportional by repo
    rng = random.Random(seed)
    parts = []
    total = len(df_in)
    for repo, g in df_in.groupby("repo", sort=False):
        k = max(1, round(n * (len(g) / total)))
        parts.append(g.sample(n=min(k, len(g)), random_state=rng.randint(0, 10**9)))
    out = pd.concat(parts, ignore_index=True)
    if len(out) > n:
        out = out.sample(n=n, random_state=seed)  # trim to exact n
    return out

def choose_subject_columns(df: pd.DataFrame):
    """Return the subject columns present (keep order preference)."""
    cols = []
    if "subject_raw" in df.columns:  cols.append("subject_raw")
    if "subject_norm" in df.columns: cols.append("subject_norm")
    return cols

# =========================
# MAIN
# =========================
def main():
    # 1) Load main list
    df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")

    # Required identifiers; 'intent' is optional (if present, we can focus on unlabeled rows)
    required_cols = {"repo", "sha"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Main_Commit_List is missing required columns: {sorted(missing)}")

    # 2) If an 'intent' column exists, select only blank-intent rows for manual labeling;
    #    otherwise use the full dataframe.
    if "intent" in df.columns:
        intent_blank_mask = df["intent"].fillna("").astype(str).str.strip().eq("")
        df_pool = df.loc[intent_blank_mask].copy()
        print(f"[INFO] Total rows: {len(df):,}")
        print(f"[INFO] Blank-intent rows (pool): {len(df_pool):,}")
    else:
        df_pool = df.copy()
        print(f"[INFO] Total rows (pool): {len(df_pool):,} (no 'intent' column present)")

    if df_pool.empty:
        raise ValueError("No rows available to sample from (pool is empty).")

    # 3) Draw overall sample
    sample_all = sample_rows(df_pool, SAMPLE_SIZE, SEED, STRATIFY_BY_REPO)

    # 4) Split into DEV/TEST (70/30) with fixed seed
    sample_all = sample_all.sample(frac=1.0, random_state=SEED).reset_index(drop=True)  # shuffle once
    n_total = len(sample_all)
    n_dev = int(round(n_total * DEV_RATIO))
    n_test = n_total - n_dev

    dev_df  = sample_all.iloc[:n_dev].copy()
    test_df = sample_all.iloc[n_dev:].copy()

    # 5) Save lean columns for review
    subject_cols = choose_subject_columns(df)
    base_cols = ["repo", "sha"]
    extra_cols = []  # add any other columns you want to see during manual review
    if "intent" in df.columns:  # keep intent if it exists for sanity checks
        extra_cols.append("intent")

    out_cols = [c for c in (base_cols + subject_cols + extra_cols) if c in sample_all.columns]

    dev_path  = OUTPUT_DIR / "DEV_Commit_Sample.csv"
    test_path = OUTPUT_DIR / "TEST_Commit_Sample.csv"

    dev_df[out_cols].to_csv(dev_path, index=False, encoding="utf-8")
    test_df[out_cols].to_csv(test_path, index=False, encoding="utf-8")

    print(f"[OK] DEV written : {dev_path}  (rows={len(dev_df)})")
    print(f"[OK] TEST written: {test_path} (rows={len(test_df)})")

    # 6) Sanity check if 'intent' exists
    if "intent" in sample_all.columns:
        leaked_dev  = (~dev_df["intent"].fillna("").astype(str).str.strip().eq("")).sum() if not dev_df.empty else 0
        leaked_test = (~test_df["intent"].fillna("").astype(str).str.strip().eq("")).sum() if not test_df.empty else 0
        print(f"[OK] Verified sampling from blank-intent only (DEV leaked={leaked_dev}, TEST leaked={leaked_test}).")

if __name__ == "__main__":
    main()


[INFO] Total rows: 106,597
[INFO] Blank-intent rows (pool): 106,597
[OK] DEV written : C:\Thesis_Temp\Second_Label_Intent\DEV_Commit_Sample.csv  (rows=700)
[OK] TEST written: C:\Thesis_Temp\Second_Label_Intent\TEST_Commit_Sample.csv (rows=300)
[OK] Verified sampling from blank-intent only (DEV leaked=0, TEST leaked=0).


## 3 - Aggregate the manual review results

In [4]:
import re
import csv
from pathlib import Path
import pandas as pd

# =========================
# CONFIG
# =========================
BASE_DIR = Path(r"C:\Thesis_Temp\Second_Label_Intent")
OUT_CSV  = BASE_DIR / "Detection_list.csv"

# Files to read: prioritize "*_with_intents.csv" but also include any CSVs present.
CANDIDATE_GLOBS = ["*_with_intents.csv", "*.csv"]

# =========================
# REGEX PATTERNS (case-insensitive)
# =========================
PATTERNS = {
    "upgrade": re.compile(r"\b(bump|upgrade|update|pin|target\s*api|agp|gradle|jdk|java\s*\d+)\b", re.IGNORECASE),
    "fix_build": re.compile(r"\b(fix|hotfix|broken|regression|fail(?:ing)?|red|green|unblock)\b", re.IGNORECASE),
    "flake_mitigation": re.compile(r"\b(flake|deflake|stabil|stability|intermittent|flaky)\b", re.IGNORECASE),
    "speed_up_ci": re.compile(r"\b(speed|faster|perf|performance|cache|parallel|shard|sharding|concurr|reduce\s*time|time\s*to\s*green)\b", re.IGNORECASE),
    "migrate_ci": re.compile(r"\b(migrat|switch|move\s*to|replace|port)\b", re.IGNORECASE),
    "cleanup": re.compile(r"\b(cleanup|tidy|refactor|format|lint)\b", re.IGNORECASE),
    "revert": re.compile(r"\b(revert|roll\s*back|back\s*out)\b", re.IGNORECASE),
}

# Intent order for the output (nice & stable)
INTENT_ORDER = [
    "upgrade",
    "fix_build",
    "flake_mitigation",
    "speed_up_ci",
    "migrate_ci",
    "cleanup",
    "revert",
]

def normalize_keyword(s: str) -> str:
    # collapse whitespace and lowercase for stable dedup/sort
    return re.sub(r"\s+", " ", s).strip().lower()

def collect_files():
    seen = set()
    files = []
    for pat in CANDIDATE_GLOBS:
        for p in BASE_DIR.glob(pat):
            if p.name.lower() == OUT_CSV.name.lower():
                continue
            if p.suffix.lower() != ".csv":
                continue
            if p.resolve() not in seen:
                seen.add(p.resolve())
                files.append(p)
    return files

def detect_from_text(df: pd.DataFrame) -> dict:
    """
    Re-run regexes on subject text to associate keywords with the correct intent.
    Returns dict[intent] -> set(keywords)
    """
    # prefer subject_norm then subject_raw; else None
    text_col = None
    if "subject_norm" in df.columns:
        text_col = "subject_norm"
    elif "subject_raw" in df.columns:
        text_col = "subject_raw"

    if text_col is None:
        return None  # caller will handle fallback

    agg = {k: set() for k in PATTERNS.keys()}

    for msg in df[text_col].fillna("").astype(str):
        for intent, rx in PATTERNS.items():
            for m in rx.finditer(msg):
                kw = normalize_keyword(m.group(1))
                agg[intent].add(kw)

    return agg

def fallback_from_existing_columns(df: pd.DataFrame) -> dict:
    """
    If subject columns are missing, best-effort: assign all row keywords to all row intents.
    This may over-attribute in a few cases, but preserves a usable summary.
    """
    if "intent_detected" not in df.columns or "keywords_detected" not in df.columns:
        return {k: set() for k in PATTERNS.keys()}

    agg = {k: set() for k in PATTERNS.keys()}
    intents_series = df["intent_detected"].fillna("").astype(str)
    kw_series      = df["keywords_detected"].fillna("").astype(str)

    for intents_str, kws_str in zip(intents_series, kw_series):
        if not intents_str:
            continue
        row_intents = [s.strip() for s in re.split(r"\s*;\s*", intents_str) if s.strip()]
        # keywords_detected was semicolon-separated in earlier code
        row_kws = [normalize_keyword(s) for s in re.split(r"\s*;\s*", kws_str) if s.strip()]
        for it in row_intents:
            if it in agg:
                for kw in row_kws:
                    agg[it].add(kw)
    return agg

def main():
    files = collect_files()
    if not files:
        raise FileNotFoundError(f"No CSV files found in {BASE_DIR}")

    # Aggregate across all files
    global_agg = {k: set() for k in PATTERNS.keys()}

    for f in files:
        try:
            df = pd.read_csv(f, encoding="utf-8-sig")
        except UnicodeDecodeError:
            df = pd.read_csv(f, encoding="utf-8", errors="ignore")

        local_agg = detect_from_text(df)
        if local_agg is None:
            # Fallback path if no subject columns exist
            local_agg = fallback_from_existing_columns(df)

        # Merge into global
        for intent, kws in local_agg.items():
            global_agg[intent].update(kws)

    # Prepare and write the summary CSV (intent, keywords)
    rows = []
    for intent in INTENT_ORDER:
        kws_sorted = sorted(global_agg[intent])  # alphabetical
        rows.append({
            "intent": intent,
            "keywords": ", ".join(kws_sorted)  # comma-separated
        })

    # Ensure folder exists (it should)
    BASE_DIR.mkdir(parents=True, exist_ok=True)

    pd.DataFrame(rows).to_csv(OUT_CSV, index=False, encoding="utf-8")
    print(f"[OK] Wrote summary: {OUT_CSV}")

if __name__ == "__main__":
    main()


[OK] Wrote summary: C:\Thesis_Temp\Second_Label_Intent\Detection_list.csv


## 3 - Review the Sample with the detection list - assign the labels to the the sample set

In [8]:
import re
from pathlib import Path
import pandas as pd

# =========================================
# CONFIG
# =========================================
BASE_DIR = Path(r"C:\Thesis_Temp\Second_Label_Intent")
DETECTION_CSV = BASE_DIR / "Detection_list.csv"   # <— use V2.0 file with regex
TEST_CANDIDATES = [
    BASE_DIR / "Test_Commit_Sample.csv",
    BASE_DIR / "TEST_Commit_Sample.csv",
]
OUT_CSV = BASE_DIR / "Test_Commit_Sample_labeled.csv"

# =========================================
# HELPERS
# =========================================
def choose_test_path(candidates):
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(
        f"Could not find Test_Commit_Sample.csv (tried: {', '.join(str(p) for p in candidates)})"
    )

def pick_text_column(df: pd.DataFrame) -> str:
    """
    Prefer subject_norm, then subject_raw, otherwise first object column.
    """
    if "subject_norm" in df.columns:
        return "subject_norm"
    if "subject_raw" in df.columns:
        return "subject_raw"
    for c in df.columns:
        if df[c].dtype == "object":
            return c
    raise ValueError("No suitable text column found to scan (need subject_norm/subject_raw or any string column).")

def load_patterns_from_detection_list(det_df: pd.DataFrame):
    """
    Read 'intent' + 'regex' from Detection_list_V2.0.csv and compile them.
    Returns a list of (intent, compiled_regex) preserving file order.
    """
    # Basic validation
    need = {"intent", "regex"}
    missing = need - set(det_df.columns.str.lower())
    # Handle case-insensitive column names
    col_map = {c.lower(): c for c in det_df.columns}
    if missing:
        raise ValueError("Detection_list_V2.0.csv must include columns: 'intent' and 'regex'")

    intents_col = col_map["intent"]
    regex_col   = col_map["regex"]

    compiled = []
    for i, row in det_df.iterrows():
        intent = str(row[intents_col]).strip()
        rx_str = str(row[regex_col]).strip()
        if not intent or not rx_str:
            continue
        try:
            # Prefer external flags; inline flags like (?i) inside rx_str are also fine.
            rx = re.compile(rx_str, re.I)
            compiled.append((intent, rx))
        except re.error as e:
            raise ValueError(f"Invalid regex for intent '{intent}': {rx_str}\n{e}")
    if not compiled:
        raise ValueError("No usable regex patterns found in Detection_list_V2.0.csv")
    return compiled

def detect_intents_for_text(text: str, compiled_patterns) -> str:
    """
    Return semicolon-separated list of intents whose patterns match the text.
    """
    s = "" if pd.isna(text) else str(text)
    hits = []
    for intent, rx in compiled_patterns:
        if rx.search(s):
            hits.append(intent)
    return "; ".join(hits)

# =========================================
# MAIN
# =========================================
def main():
    # Load detection list (must have intent + regex)
    try:
        det_df = pd.read_csv(DETECTION_CSV, encoding="utf-8-sig")
    except UnicodeDecodeError:
        det_df = pd.read_csv(DETECTION_CSV, encoding="utf-8", errors="ignore")
    compiled_patterns = load_patterns_from_detection_list(det_df)

    # Load TEST sample
    test_path = choose_test_path(TEST_CANDIDATES)
    try:
        test_df = pd.read_csv(test_path, encoding="utf-8-sig")
    except UnicodeDecodeError:
        test_df = pd.read_csv(test_path, encoding="utf-8", errors="ignore")

    text_col = pick_text_column(test_df)

    # Detect intents
    test_df["Intent_Detected"] = (
        test_df[text_col]
        .fillna("")
        .astype(str)
        .apply(lambda s: detect_intents_for_text(s, compiled_patterns))
    )

    # Save
    OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    test_df.to_csv(OUT_CSV, index=False, encoding="utf-8")
    print(f"[OK] Labeled file written: {OUT_CSV}")

    # Optional: quick summary
    counts = (
        test_df["Intent_Detected"]
        .str.split(r"\s*;\s*")
        .explode()
        .dropna()
        .loc[lambda s: s.str.len() > 0]
        .value_counts()
    )
    if not counts.empty:
        print("\nPredicted counts by intent:")
        for intent, n in counts.items():
            print(f"  {intent}: {n}")

if __name__ == "__main__":
    main()


[OK] Labeled file written: C:\Thesis_Temp\Second_Label_Intent\Test_Commit_Sample_labeled.csv

Predicted counts by intent:
  upgrade: 113
  fix_build: 37
  speed_up_ci: 5
  cleanup: 5
  migrate_ci: 3
  flake_mitigation: 1
  revert: 1


## 4 - Measure the detection's accuracy

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path(r"C:\Thesis_Temp\Second_Label_Intent")
IN_CSV  = BASE / "Test_Commit_Sample_labeled_manual.csv"  # must have Intent_Detected and intent_Manual
OUT_CSV = BASE / "Test_Intent_Metrics.csv"

def parse_labels(val):
    if pd.isna(val): return set()
    s = str(val).strip()
    if not s: return set()
    return set(p.strip().lower() for p in s.split(";") if p.strip())

df = pd.read_csv(IN_CSV, encoding="utf-8-sig")
y_true_sets = df.get("intent_Manual", "").apply(parse_labels)
y_pred_sets = df.get("Intent_Detected", "").apply(parse_labels)

all_labels = sorted(set().union(*y_true_sets, *y_pred_sets))

rows = []
micro_tp = micro_fp = micro_fn = 0
for label in all_labels:
    tp = fp = fn = tn = 0
    for yt, yp in zip(y_true_sets, y_pred_sets):
        yt_has = label in yt
        yp_has = label in yp
        if yt_has and yp_has: tp += 1
        elif not yt_has and yp_has: fp += 1
        elif yt_has and not yp_has: fn += 1
        else: tn += 1
    micro_tp += tp; micro_fp += fp; micro_fn += fn
    prec = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    rec  = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    f1   = (2*prec*rec)/(prec+rec) if np.isfinite(prec) and np.isfinite(rec) and (prec+rec)>0 else np.nan
    rows.append({
        "intent": label,
        "support (manual positives)": tp + fn,
        "predicted positives": tp + fp,
        "TP": tp, "FP": fp, "FN": fn,
        "precision": prec, "recall": rec, "f1": f1,
    })

report = pd.DataFrame(rows).sort_values("intent").reset_index(drop=True)

macro_precision = report["precision"].mean(skipna=True)
macro_recall    = report["recall"].mean(skipna=True)
macro_f1        = report["f1"].mean(skipna=True)

micro_precision = micro_tp / (micro_tp + micro_fp) if (micro_tp + micro_fp) > 0 else np.nan
micro_recall    = micro_tp / (micro_tp + micro_fn) if (micro_tp + micro_fn) > 0 else np.nan
micro_f1        = (2*micro_precision*micro_recall)/(micro_precision+micro_recall) if np.isfinite(micro_precision) and np.isfinite(micro_recall) and (micro_precision+micro_recall)>0 else np.nan

summary = pd.DataFrame([
    {"intent": "MACRO_AVG", "precision": macro_precision, "recall": macro_recall, "f1": macro_f1},
    {"intent": "MICRO_AVG", "precision": micro_precision, "recall": micro_recall, "f1": micro_f1,
     "TP": micro_tp, "FP": micro_fp, "FN": micro_fn},
])

final = pd.concat([report, summary], ignore_index=True)
final.to_csv(OUT_CSV, index=False, encoding="utf-8")
print(f"[OK] wrote {OUT_CSV}")


[OK] wrote C:\Thesis_Temp\Second_Label_Intent\Test_Intent_Metrics.csv


## update the main list

In [28]:
import re
from pathlib import Path
from typing import List, Dict
import pandas as pd

# =========================
# CONFIGURE THESE PATHS
# =========================
BASE_DIR = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label\Method_V2.0")
MAIN_COMMIT_LIST = BASE_DIR / "Main_Commit_List.csv"                # input
DETECTION_LIST   = BASE_DIR / "detection_list.csv"                  # input
OUTPUT_FILE      = BASE_DIR / "Main_Commmit_List_Updated.csv"       # output (3 m's)

SEP = "|"  # joiner for multi-values

def split_keywords_colon(s: str) -> List[str]:
    """Split detection_list keywords by colon only, trim, drop empties."""
    if not isinstance(s, str):
        return []
    # allow spaces around colons, tolerate accidental quotes
    parts = [p.strip().strip('"').strip("'") for p in s.split(":")]
    return [p for p in parts if p]

def choose_text_column(df: pd.DataFrame) -> str:
    """Choose the commit text column to search."""
    for col in ["subject_norm", "subject_raw", "subject", "message"]:
        if col in df.columns:
            return col
    df["subject_norm"] = ""  # fallback
    return "subject_norm"

def main():
    # Read inputs (utf-8-sig tolerates BOM from Excel)
    df = pd.read_csv(MAIN_COMMIT_LIST, encoding="utf-8-sig")
    det = pd.read_csv(DETECTION_LIST,   encoding="utf-8-sig")

    # Normalize detector headers & filter usable rows
    det.columns = [c.strip().lower() for c in det.columns]
    if "intent" not in det.columns or "keywords" not in det.columns:
        raise ValueError("detection_list.csv must have 'intent' and 'keywords' columns.")

    if "enabled" in det.columns:
        det["enabled"] = (
            det["enabled"].fillna(True).astype(str).str.strip().str.lower()
            .isin(["1","true","t","yes","y"])
        )
        det = det[det["enabled"]]

    det = det[(det["intent"].astype(str).str.strip() != "") &
              (det["keywords"].astype(str).str.strip() != "")].copy()

    # Expand to (intent, keyword) rows using COLON splitting
    rows: List[Dict[str, str]] = []
    for _, r in det.iterrows():
        intent = str(r["intent"]).strip()
        for kw in split_keywords_colon(r["keywords"]):
            rows.append({"intent": intent, "keyword": kw})
    if not rows:
        # no detectors; write blank columns and exit
        df["intent"] = ""
        df["keywords"] = ""
        df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")
        print(f"[WARN] No usable detectors. Wrote {OUTPUT_FILE} with empty columns.")
        return

    kw_df = pd.DataFrame(rows).drop_duplicates()
    kw_df["kw_lc"] = kw_df["keyword"].astype(str).str.lower()

    # Choose text column and lower-case it for matching
    text_col = choose_text_column(df)
    text_lc = df[text_col].fillna("").astype(str).str.lower()

    intents_out, keywords_out = [], []

    # Brute-force substring matching (case-insensitive)
    kws = list(zip(kw_df["intent"].to_list(), kw_df["keyword"].to_list(), kw_df["kw_lc"].to_list()))
    for txt in text_lc.tolist():
        matched_intents = set()
        matched_keywords = set()
        for intent, kw_orig, kw_lc in kws:
            if kw_lc and kw_lc in txt:
                matched_intents.add(intent)
                matched_keywords.add(kw_orig)  # keep original casing
        intents_out.append(SEP.join(sorted(matched_intents)) if matched_intents else "")
        keywords_out.append(SEP.join(sorted(matched_keywords)) if matched_keywords else "")

    # Attach and save
    df["intent"] = intents_out
    df["keywords"] = keywords_out
    df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")
    print(f"[OK] Wrote {OUTPUT_FILE} with columns: intent, keywords")

if __name__ == "__main__":
    main()


[OK] Wrote C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label\Method_V2.0\Main_Commmit_List_Updated.csv with columns: intent, keywords
